# 01 — Introduction, Problem Definition & Data Loading
**Event Recommendation System** · Stages 1 & 2

Define the problem we are solving, then load the dataset and inspect what is *actually* in it before writing any model code.

## 1. The problem

Event platforms list hundreds of events — concerts, workshops, hikes, meetups. Most users only care about a few of them, and scrolling through everything is slow. Generic "popular events" lists ignore that one person loves tech meetups in Cairo while another wants free yoga in Alexandria.

**Goal:** for each user, rank the upcoming events so the ones they will actually go to appear in the top 10.

### Why events are harder than movies or products
| Challenge | What it means | How we handle it |
|---|---|---|
| **Item cold start** | An event exists for a few weeks, then it is over. Most upcoming events have little or no interaction history. | Content-based filtering reads the event's text, category, city and price — it works from day one. |
| **User cold start** | New users have no history. | Use the interests they selected at sign-up + their city. |
| **Sparsity** | Each user touches ~3% of events. | Matrix factorisation (SVD) learns compact "taste" vectors from sparse data. |
| **Time** | We must predict the *future* from the *past*. | A strict time-based train / validation / test split. |

### Our approach
1. **Content-Based Filtering (CBF)** — "you liked events *like this one*": TF-IDF on event text + category/city/price features, compared with a user profile using cosine similarity.
2. **Collaborative Filtering (CF) with SVD** — "people *like you* liked this": learns hidden taste factors from everyone's behaviour.
3. **Hybrid** — combines both, trusting CF more when an event has enough history and CBF when it is new.

### Two prediction tasks, five metrics
| Task | Question | Metrics |
|---|---|---|
| **Rating prediction** | How many stars would this user give this event? | MAE, RMSE |
| **Top-10 ranking** | Which 10 upcoming events should we show? | Precision@10, Recall@10, NDCG@10 |

> **About the data:** no public event dataset contains user interests, event descriptions and ratings together, so this project uses a **synthetic dataset** generated by a behaviour simulation (`00_generate_dataset.ipynb`). The models never see the hidden tastes used by the simulator — only the CSV files.

### Setup (run this first)
- **In Google Colab** this cell downloads the project from GitHub (data + `src/` code) and moves into it.
- **On your own computer** it just moves from `notebooks/` to the project root.

`sys.path.insert(0, "src")` lets us `import recsys` - the small package that holds the reusable code (cleaning, models, metrics). Notebooks *explain* the code; `src/recsys` *stores* it so every notebook uses exactly the same implementation.

In [1]:
import os, sys, subprocess

REPO_URL = "https://github.com/Abdelrahman2115/recommender-system.git"

if "google.colab" in sys.modules:                       # running inside Google Colab
    if not os.path.exists("/content/recommender-system"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/recommender-system"], check=True)
    os.chdir("/content/recommender-system")
elif os.path.basename(os.getcwd()) == "notebooks":      # running locally from the notebooks folder
    os.chdir("..")

sys.path.insert(0, "src")                               # make `import recsys` work
print("Project root:", os.getcwd())

Project root: /home/claude/recommender-system


## 2. Load the data
`pd.read_csv` reads each file into a DataFrame (a table). We do **not** parse dates yet — first we want to see the raw file exactly as it is, then decide how to treat each column.

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 20)       # show all columns when printing tables
pd.set_option("display.width", 140)

users = pd.read_csv("data/raw/users.csv")
events = pd.read_csv("data/raw/events.csv")
interactions = pd.read_csv("data/raw/interactions.csv")

for name, df in [("users", users), ("events", events), ("interactions", interactions)]:
    print(f"{name:13s} rows = {len(df):>6,}   columns = {df.shape[1]}")

users         rows =  2,000   columns = 5
events        rows =    800   columns = 10
interactions  rows = 50,518   columns = 5


### 2.1 First rows of each table
`head()` shows the first rows — the quickest way to understand what a table contains.

In [3]:
users.head()

,user_id,age,city,interests,signup_date
0,1,30.0,Giza,Food & Drink|Arts & Culture,2025-03-31
1,2,22.0,Alexandria,Food & Drink|Sports & Fitness|Education & Workshops,2024-12-02
2,3,29.0,Cairo,Arts & Culture|Sports & Fitness,2025-07-14
3,4,16.0,Cairo,Music|Outdoors & Travel|Arts & Culture,2025-05-05
4,5,26.0,Giza,Education & Workshops|Music,2024-05-15


In [4]:
events.head(3)

,event_id,title,description,category,tags,city,venue_type,start_datetime,price,organizer_id
0,1,Language Exchange Gathering,"Language Exchange Gathering. A languages event about no experience needed, english, egypt, learn, german, french, session.",Education & Workshops,conversation|german,Alexandria,cafe,2025-07-21 19:00:00,260.0,52
1,2,Brewing Gathering,"Brewing Gathering. A coffee event about barista, brewing, special, premium, for professionals, web development, fun, session, roasting.",Food & Drink,coffee|barista,Cairo,hall,2024-12-01 10:00:00,120.0,28
2,3,Ethical Hacking Night,"Ethical Hacking Night. A cybersecurity event about evening, join us, cloud, advanced, privacy, friends, ethical hacking.",Technology,networks|ethical hacking,Cairo,hall,2024-10-23 20:00:00,80.0,11


In [5]:
interactions.head()

,user_id,event_id,timestamp,interaction_type,rating
0,312,226,2024-01-19 23:58:00,view,NaN
1,511,242,2024-01-21 09:08:00,view,NaN
2,470,72,2024-01-26 08:22:00,view,NaN
3,1276,170,2024-01-26 11:03:00,view,NaN
4,470,614,2024-01-26 18:02:00,going,NaN


### 2.2 Column types
`dtypes` tells us how pandas stored each column. Two things to notice:
- Dates (`signup_date`, `start_datetime`, `timestamp`) were read as plain text (`object`/`str`) — we must convert them to real datetimes before doing any time logic.
- `age` and `rating` are floats even though they look like integers: pandas uses float because those columns contain missing values (`NaN` is a float).

In [6]:
for name, df in [("users", users), ("events", events), ("interactions", interactions)]:
    print(f"--- {name} ---")
    print(df.dtypes.to_string(), "\n")

--- users ---
user_id          int64
age            float64
city               str
interests          str
signup_date        str 

--- events ---
event_id            int64
title                 str
description           str
category              str
tags                  str
city                  str
venue_type            str
start_datetime        str
price             float64
organizer_id        int64 

--- interactions ---
user_id               int64
event_id              int64
timestamp               str
interaction_type        str
rating              float64 



### 2.3 Missing values
`isna().sum()` counts empty cells per column. A missing `rating` is **expected** (only attended events get rated); missing ages, descriptions and prices are data-quality problems to fix in Stage 4.

In [7]:
missing = pd.concat({
    "users": users.isna().sum(),
    "events": events.isna().sum(),
    "interactions": interactions.isna().sum(),
}).rename("missing").to_frame()
missing[missing["missing"] > 0]

missing
users        age               60
events       description       12
             price             10
interactions rating         39657

### 2.4 Keys and relationships
A recommender joins three tables: `interactions.user_id → users.user_id` and `interactions.event_id → events.event_id`. We check:
1. IDs are unique in `users` and `events` (so they can be primary keys),
2. every interaction points to an existing user and event (no "orphan" rows),
3. how many exact duplicate rows exist.

In [8]:
print("user_id unique in users:  ", users["user_id"].is_unique)
print("event_id unique in events:", events["event_id"].is_unique)
print("interactions with unknown user: ", (~interactions["user_id"].isin(users["user_id"])).sum())
print("interactions with unknown event:", (~interactions["event_id"].isin(events["event_id"])).sum())
print("exact duplicate interaction rows:", interactions.duplicated().sum())
print("duplicate (user, event) pairs:   ", interactions.duplicated(["user_id", "event_id"]).sum())

user_id unique in users:   True
event_id unique in events: True
interactions with unknown user:  0
interactions with unknown event: 0
exact duplicate interaction rows: 500
duplicate (user, event) pairs:    500


### 2.5 Categorical columns
`value_counts()` lists the distinct values of a column and how often each appears. This is how we discover problems such as the same city written in several ways.

In [9]:
print(interactions["interaction_type"].value_counts(), "\n")
print(events["category"].value_counts(), "\n")
print(events["venue_type"].value_counts())

interaction_type
view          22174
interested    14692
attended      10861
going          2791
Name: count, dtype: int64 

category
Music                    154
Arts & Culture           123
Education & Workshops    105
Food & Drink             100
Outdoors & Travel        100
Sports & Fitness          85
Business & Career         69
Technology                64
Name: count, dtype: int64 

venue_type
hall                190
cafe                170
university          123
outdoor             121
co-working space    115
online               81
Name: count, dtype: int64


In [10]:
# City spelling: the same city appears in several forms (case and spaces)
users["city"].value_counts()

,count
city,
Cairo,773
Giza,384
Alexandria,344
Mansoura,170
Tanta,107
Hurghada,103
Luxor,79
CAIRO,7
cairo,6


### 2.6 Numeric columns
`describe()` gives count, mean, spread and min/max. Look at `age`: a minimum of 0 and a maximum of 999 are impossible — typing mistakes we must clean.

In [11]:
users[["age"]].describe().T

,count,mean,std,min,25%,50%,75%,max
age,1940.0,27.425773,23.706673,0.0,22.0,26.0,31.0,999.0


In [12]:
events[["price"]].describe().T

,count,mean,std,min,25%,50%,75%,max
price,790.0,188.468354,198.705719,0.0,0.0,140.0,260.0,1290.0


### 2.7 Ratings: only for attended events
This checks a business rule: a user can only rate an event they attended.

In [13]:
rated = interactions.dropna(subset=["rating"])
print(rated["interaction_type"].value_counts())
print("\nRating values:", sorted(rated["rating"].unique()))

interaction_type
attended    10861
Name: count, dtype: int64

Rating values: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]


### 2.8 Time range
We convert the text dates with `pd.to_datetime` to see the period the data covers.

In [14]:
ts = pd.to_datetime(interactions["timestamp"])
starts = pd.to_datetime(events["start_datetime"])
print("Interactions from", ts.min(), "to", ts.max())
print("Events start from", starts.min(), "to", starts.max())

Interactions from 2024-01-19 23:58:00 to 2025-12-31 00:00:00
Events start from 2024-01-31 12:00:00 to 2025-12-30 10:00:00


## 3. Data dictionary (from what we just inspected)

**users.csv** — 2,000 rows
| Column | Type | Meaning | Issues found |
|---|---|---|---|
| user_id | int | unique user ID | — |
| age | float | age in years | 60 missing, impossible values (0, 999…) |
| city | text | home city (7 Egyptian cities) | inconsistent case / spaces |
| interests | text | categories chosen at sign-up, separated by `\|` | — |
| signup_date | date | when the account was created | stored as text |

**events.csv** — 800 rows
| Column | Type | Meaning | Issues found |
|---|---|---|---|
| event_id | int | unique event ID | — |
| title, description | text | event text (used by TF-IDF) | 12 missing descriptions |
| category | text | one of 8 categories | — |
| tags | text | 2 keywords, separated by `\|` | — |
| city, venue_type | text | where it happens (`online` is possible) | — |
| start_datetime | date | when the event starts | stored as text |
| price | float | ticket price in EGP (0 = free) | 10 missing |
| organizer_id | int | who organises it | — |

**interactions.csv** — 50,518 rows
| Column | Type | Meaning | Issues found |
|---|---|---|---|
| user_id, event_id | int | who interacted with what | — |
| timestamp | date | when it happened | stored as text |
| interaction_type | text | view < interested < going < attended | — |
| rating | float | 1–5 stars, only when attended | missing for non-attended rows (expected) |

Table-level issue: **500 exact duplicate rows** (the same interaction logged twice).

### Interview questions for this stage
- *What is the difference between explicit and implicit feedback?* — Ratings are **explicit** (the user tells us their opinion). Views, "interested" and "going" are **implicit** (we infer interest from behaviour). Implicit data is plentiful but noisy; explicit data is precise but rare — here only ~21% of interactions have a rating.
- *Why inspect the data before modelling?* — So the code follows the real columns. For example, we found that ratings only exist for attended events, which decides how we evaluate rating prediction.

**Next:** `02_eda.ipynb` — explore the data visually.